# Here is an example of loeding a model from Azure

In [ ]:
from azure_api import Client 
api_version="2024-12-01-preview"
model_name="gpt-5-mini"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [11]:
msg = [
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ]
content, raw = model.send_msg_and_get_contnent(msg)

In [12]:
print(content)

Great — Paris has a lot to offer. Here are concise recommendations so you can pick what fits your time and interests.

Top must-sees (quick list)
- Eiffel Tower (view from Trocadéro; reserve summit tickets if you want to go up)
- Louvre (Mona Lisa, Egyptian and Greek collections; buy timed tickets)
- Musée d’Orsay (Impressionists)
- Notre-Dame area / Île de la Cité and Sainte‑Chapelle (stained glass)
- Montmartre & Sacré‑Cœur (artists’ square, panoramic views)
- Arc de Triomphe & Champs‑Élysées (climb the arch for views)
- Le Marais (historic streets, shops, cafés, Jewish quarter)
- Latin Quarter & Panthéon (student vibe, narrow streets)
- Luxembourg Gardens (relaxation and people-watching)
- Seine river cruise (especially at sunset)
- Centre Pompidou (modern art + rooftop view)
- Musée Rodin (sculptures in a lovely garden)
- Pere Lachaise Cemetery (Jim Morrison, Oscar Wilde)
- Musée de l’Orangerie (Monet’s Water Lilies)
- Versailles (day trip; palace and gardens — reserve tickets)
- M

In [12]:
from dotenv import load_dotenv
import os
import requests
from datasets import load_dataset
from math_arena_datasets import categories_2025
import pandas as pd
from sklearn.model_selection import train_test_split
from azure_api import Client 
from time import sleep
from openai import AzureOpenAI
import numpy as np
import ast

# client = OpenAI()

# --- Load API key ---
load_dotenv()
# API_URL = "https://api.groq.com/openai/v1/chat/completions"
# API_KEY = os.environ.get("GROQ_API_KEY")

# --- Build the few-shot prompt ---
def build_prompt(examples, query):
    categories = [
        "Arithmetic", "Algebra", "Geometry",
        "Combinatorics", "Number Theory", "Probability"
    ]
    
    prompt = (
        "You are a math problem classifier.\n"
        f"Your task is to assign each problem to exactly one of these categories:\n"
        f"{', '.join(categories)}.\n"
        "Respond with only the category name.\n\n"
    )
    
    for _, row in examples.iterrows():
        prompt += f"### Example\nProblem: {row['problem']}\nCategory: {row['problem_type']}\n\n"
    
    prompt += f"### Classify this new problem\nProblem: {query}\nCategory:"
    return prompt

# --- Query the Groq API ---
def classify_with_azure(prompt):
    client = AzureOpenAI(
        api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
        api_version="2024-12-01-preview",
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    )

    try:
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}],
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print("Error:", e)
        return ""

# --- Load and merge all datasets ---
dfs = []
for path in categories_2025:
    dataset_dict = load_dataset(path)
    split = list(dataset_dict.keys())[0]
    df_split = dataset_dict[split].to_pandas()
    dfs.append(df_split)

df = pd.concat(dfs, ignore_index=True)
df["problem_type"] = df["problem_type"].apply(lambda x: x[0] if isinstance(x, list) else x)

# Flatten the 'problem_type' column to strings
def flatten_label_from_csv(x):
    try:
        # Convert string representation of list to actual list
        lst = ast.literal_eval(x)
        if isinstance(lst, list) and len(lst) > 0:
            return str(lst[0]).strip()
    except:
        pass
    return str(x).strip()  # fallback if not a list


# --- Keep only needed columns ---
df = df[["problem", "problem_type"]].dropna()

# --- Train/test split ---
train_df, test_df = pd.read_csv('data/problem_type_data.train.csv'), pd.read_csv('data/problem_type_data.test.csv')
# Flatten labels
train_df['problem_type'] = train_df['problem_type'].apply(flatten_label_from_csv)
test_df['problem_type'] = test_df['problem_type'].apply(flatten_label_from_csv)

# --- Evaluate with few-shot ICL ---
correct = 0  # <-- initialize counter

for _, row in test_df.iterrows():
    few_shots = train_df.sample(5)
    prompt = build_prompt(few_shots, row["problem"])
    predicted = classify_with_azure(prompt)

    # Clean predicted text
    predicted = predicted.strip().replace("Category:", "").strip()

    actual = row['problem_type']  # already a string now
    print(f"Predicted: {predicted} | Actual: {actual}")

    if predicted.lower() == actual.lower():
        correct += 1

accuracy = correct / len(test_df)
print(f"\nAccuracy: {accuracy:.2%}")




Predicted: Geometry | Actual: Geometry
Predicted: Combinatorics | Actual: Combinatorics
Predicted: Number Theory | Actual: Algebra
Predicted: Combinatorics | Actual: Combinatorics
Predicted: Geometry | Actual: Algebra
Predicted: Geometry | Actual: Algebra
Predicted: Geometry | Actual: Geometry
Predicted: Number Theory | Actual: Number Theory
Predicted: Combinatorics | Actual: Combinatorics
Predicted: Geometry | Actual: AlgebraGeometry
Predicted: Probability | Actual: Number Theory
Predicted: Algebra | Actual: Algebra
Predicted: Algebra | Actual: Algebra
Predicted: Algebra | Actual: Algebra
Predicted: Number Theory | Actual: Algebra
Predicted: Algebra | Actual: AlgebraNumber Theory
Predicted: Combinatorics | Actual: Combinatorics
Predicted: Combinatorics | Actual: Combinatorics
Predicted: Combinatorics | Actual: Combinatorics
Predicted: Probability | Actual: CombinatoricsGeometry
Predicted: Geometry | Actual: Geometry
Predicted: Geometry | Actual: Geometry
Predicted: Geometry | Actual: 